#Unofficial [LeetArxiv](https://leetarxiv.substack.com/p/gumbel-sinkhorn-neural-sort) implementation of the paper *Learning Latent Permutations with Gumbel-Sinkhorn Networks*

The 2018 paper *Learning Latent Permutations with Gumbel-Sinkhorn Network* (Mena et al., 2018) introduces *Gumbel-Sinkhorn networks*: neural networks that learn to sort items by differentiation.


The complete writeup is available [here](https://leetarxiv.substack.com/p/gumbel-sinkhorn-neural-sort)

Gumbel-Sinkhorn networks build on the fact that permutations can be represented by matrices.
The authors are further inspired by:
1. The [reparametarization trick](https://leetarxiv.substack.com/p/stable-diffusion-from-scratch-1?utm_source=publication-search) from diffusion models.
2. The [1967 Sinkhorn matrix](https://leetarxiv.substack.com/p/sinkhorn-knopp-algorithm-24d?utm_source=publication-search) balancing algorithm.
3. [Belief propagation](https://leetarxiv.substack.com/p/sinkhorn-solves-sudoku?utm_source=publication-search) an obscure alternative to backprop that uses Sinkhorn balancing to solve Sudoku.

First, we’ll code two fundamental concepts introduced in (Mena et al., 2018):

1.   **Sinkhorn operator**: a differentiable approximation of a permutation.
2.   **Gumbel-Sinkhorn distribution**: an analog of the Gumbel Softmax distribution built on the [reparameterization trick](https://leetarxiv.substack.com/p/stable-diffusion-from-scratch-1?utm_source=publication-search) like in stable diffusion.








## Sinkhorn Operator
This is section 2.1 in the [LeetArxiv article](https://leetarxiv.substack.com/p/gumbel-sinkhorn-neural-sort).

The Sinkhorn operator is simply the [1967 Sinkhorn matrix](https://leetarxiv.substack.com/p/sinkhorn-knopp-algorithm-24d?utm_source=publication-search) balancing algorithm but divisions are replaced with logarithms and subtraction (for numerical stability).

We test that the row and cols sum to 1 as expected.

In [1]:
import torch
def SinkhornOperator(squareMatrix, sinkhornIterations):
    """
    Args:
      squareMatrix: a square matrix of shape [N, N], or a tensor of [batchSize, N, N]
      sinkhornIterations: number of sinkhorn iterations, usually ~20
    Returns:
      A 3D tensor of close-to-doubly-stochastic matrices (2D tensors are
        converted to 3D tensors with batchSizeequals to 1)
    """
    assert squareMatrix.shape[-1] == squareMatrix.shape[-2], \
        f"Last two dimensions must be equal, got {tuple(squareMatrix.shape)}"
    for iteration in range(sinkhornIterations):
        squareMatrix = squareMatrix - torch.logsumexp(squareMatrix, -1, keepdim=True)
        squareMatrix = squareMatrix - torch.logsumexp(squareMatrix, -2, keepdim=True)
    return squareMatrix.exp() #Remove logs

def TestSinkhornOperator():
  #Create a random square matrix.
  matrixDimension = 10
  sinkhornIterations = 12
  squareMatrix = torch.rand((matrixDimension, matrixDimension))
  #Apply the sinkhorn operator
  sinkhornResult = SinkhornOperator(torch.log(squareMatrix), sinkhornIterations)
  #Ensure all rows sum to 1.
  assert torch.allclose(sinkhornResult.sum(dim=0), torch.ones(sinkhornResult.shape[0]))
  #Ensure all columns sum to 1.
  assert torch.allclose(sinkhornResult.sum(dim=1), torch.ones(sinkhornResult.shape[1]))
  print(f"Original matrix :\n{squareMatrix}")
  print(f"Balanced matrix :\n{sinkhornResult}")

if __name__ == "__main__":
    TestSinkhornOperator()


Original matrix :
tensor([[0.9193, 0.4841, 0.2690, 0.4109, 0.0410, 0.5830, 0.1773, 0.8858, 0.4131,
         0.4117],
        [0.7539, 0.9699, 0.1297, 0.4244, 0.0143, 0.7456, 0.7326, 0.9482, 0.0467,
         0.3617],
        [0.7512, 0.4519, 0.9511, 0.1786, 0.5573, 0.2412, 0.1816, 0.5937, 0.5154,
         0.4044],
        [0.2633, 0.0173, 0.5177, 0.0839, 0.0972, 0.0420, 0.6090, 0.0061, 0.0976,
         0.0729],
        [0.0466, 0.7809, 0.8654, 0.6849, 0.7026, 0.1358, 0.2621, 0.1361, 0.7453,
         0.5054],
        [0.0145, 0.5211, 0.7718, 0.5286, 0.2575, 0.0819, 0.3173, 0.4729, 0.1602,
         0.4606],
        [0.2826, 0.5634, 0.0112, 0.7905, 0.5952, 0.2733, 0.5657, 0.9208, 0.2423,
         0.3209],
        [0.6219, 0.2569, 0.6724, 0.8092, 0.7797, 0.2693, 0.0469, 0.4398, 0.8461,
         0.0715],
        [0.2736, 0.9763, 0.0690, 0.5453, 0.4986, 0.3808, 0.1056, 0.4700, 0.9114,
         0.4094],
        [0.0953, 0.0465, 0.2835, 0.9542, 0.4467, 0.1158, 0.0180, 0.3285, 0.1519,
         0

## Gumbel-Sinkhorn Distribution
This is section 2.2 in the [LeetArxiv article](https://leetarxiv.substack.com/p/gumbel-sinkhorn-neural-sort).

The *Gumbel-Sinkhorn distribution* is the distribution obtained by applying the Sinkhorn-operator on an unnormalized assignment probability matrix to which we add Gumbel noise.

The concept's borrowed from Stable Diffusion's [reparameterization trick](https://leetarxiv.substack.com/p/stable-diffusion-from-scratch-1?utm_source=publication-search).


In [2]:
def GenerateGumbelNoise(tensorShape, device='cpu', eps=1e-20):
    """
    Args:
      tensorShape: list of tensor dimensions
      eps: small float to avoid division by zero
    """
    noise = torch.rand(tensorShape, device=device)
    return -torch.log(-torch.log(noise + eps) + eps)

def GumbelSinkhorn(sinkhornProbabilities, tau, sinkhornIterations):
    """Generate a Gumbel-Sinkhorn matrix at timestep tau.
    Args:
      sinkhornProbabilities: square matrix of log probabilities obtained from SinkhornOperator.
      tau: Temperature parameter. Think of it like the timestep parameter in diffusion forward process
           Small tau is little noise. Big tau is super noisy.
    """
    gumbelNoise = GenerateGumbelNoise(sinkhornProbabilities.shape, device=sinkhornProbabilities.device)
    #Apply the Sinkhorn operator
    noiseAtTimestepTau = SinkhornOperator((sinkhornProbabilities + gumbelNoise)/tau, sinkhornIterations)
    return noiseAtTimestepTau

def TestTau():
  #Create a random square matrix.
  matrixDimension = 4
  sinkhornIterations = 20
  squareMatrix = torch.rand((matrixDimension, matrixDimension), requires_grad=True)
  #Generate noisy versions
  smallTauResult = GumbelSinkhorn(squareMatrix, tau=0.01, sinkhornIterations=sinkhornIterations)
  bigTauResult   = GumbelSinkhorn(squareMatrix, tau=11,   sinkhornIterations=sinkhornIterations)

  #Test gradients
  X = torch.rand((matrixDimension, matrixDimension), requires_grad=True)
  P = GumbelSinkhorn(X*1000, tau=18, sinkhornIterations=2000)
  P.sum().backward()
  print(f"smallTauResult : Matrix entries are close to whole numbers and resemble a permutation matrix somewhat\n{smallTauResult}\n")
  print(f"bigTauResult   : Barely  resembles a permutation matrix\n{bigTauResult}\n")
  print(f"X Gradients    : It has gradients (lol you might see zeros bcoz the grads are super small)\n{X.grad}\n")

if __name__ == "__main__":
    TestTau()

smallTauResult : Matrix entries are close to whole numbers and resemble a permutation matrix somewhat
tensor([[4.9939e-01, 4.8855e-03, 0.0000e+00, 4.6541e-23],
        [0.0000e+00, 0.0000e+00, 1.0000e+00, 0.0000e+00],
        [5.0061e-01, 0.0000e+00, 0.0000e+00, 0.0000e+00],
        [9.1173e-13, 9.9511e-01, 0.0000e+00, 1.0000e+00]],
       grad_fn=<ExpBackward0>)

bigTauResult   : Barely  resembles a permutation matrix
tensor([[0.2549, 0.2383, 0.1834, 0.3234],
        [0.2806, 0.2516, 0.2388, 0.2289],
        [0.2119, 0.2534, 0.3242, 0.2105],
        [0.2526, 0.2566, 0.2535, 0.2372]], grad_fn=<ExpBackward0>)

X Gradients    : It has gradients (lol you might see zeros bcoz the grads are super small)
tensor([[ 2.4845e-20,  6.9431e-17, -4.0782e-08,  4.0782e-08],
        [ 8.0024e-09,  2.3513e-09, -1.0354e-08,  4.1489e-19],
        [-3.6049e-20, -1.8061e-09,  5.3367e-14,  1.8061e-09],
        [-8.0024e-09, -5.4521e-10,  5.1135e-08, -4.2588e-08]])



## Gumbel-Sinkhorn Networks
This is section 2.3 in the [LeetArxiv article](https://leetarxiv.substack.com/p/gumbel-sinkhorn-neural-sort).

*Gumbel-Sinkhorn networks* build on the Sinkhorn-operator and Gumbel-Sinkhorn sampler.


In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

def matching(logAlpha):
  """Greedy assignment for evaluation"""
  probs = torch.exp(logAlpha)
  n = logAlpha.shape[0]
  assignment = torch.zeros_like(probs)
  for i in range(n):
    remainingCols = (assignment.sum(dim=0) == 0)
    if remainingCols.sum() == 0:
      break
    best_col = torch.argmax(probs[i] * remainingCols.float())
    assignment[i, best_col] = 1
  return assignment

class NumberGenerator(Dataset):
  def __init__(self, numberListLength, noOfLists, minValue=0, maxValue=1, seed=1):
    self.numberListLength = numberListLength
    torch.manual_seed(seed)
    self.X = torch.zeros(noOfLists, numberListLength).uniform_(minValue, maxValue)
    self.sortedX, self.permutation = self.X.sort(1)

  def __getitem__(self, idx):
    return self.X[idx], self.sortedX[idx], self.permutation[idx]

  def __len__(self):
    return len(self.X)

class NumberSortingNetwork(nn.Module):
  def __init__(self, numberListLength, tau=1.0, sinkhornIterations=20):
    super().__init__()
    self.numberListLength = numberListLength
    self.tau = tau
    self.sinkhornIterations = sinkhornIterations

    self.encoder = nn.Sequential(
      nn.Linear(numberListLength, numberListLength * 2),
      nn.ReLU(),
      nn.Linear(numberListLength * 2, numberListLength * 4),
      nn.ReLU(),
      nn.Linear(numberListLength * 4, numberListLength * numberListLength)
    )

  def forward(self, x):
    batchSize = x.shape[0]
    logitsFlattened = self.encoder(x)
    logits = logitsFlattened.view(batchSize, self.numberListLength, self.numberListLength)
    probs = torch.exp(logits - torch.logsumexp(logits, dim=-1, keepdim=True))

    if self.training:
      permutationMatrices = []
      for i in range(batchSize):
        P = GumbelSinkhorn(probs[i], tau=self.tau, sinkhornIterations=self.sinkhornIterations)
        permutationMatrices.append(P)
      P = torch.stack(permutationMatrices)
    else:
      permutationMatrices = []
      for i in range(batchSize):
        P = matching(logits[i])
        permutationMatrices.append(P)
      P = torch.stack(permutationMatrices)
    sortedX = torch.bmm(P, x.unsqueeze(-1)).squeeze(-1)
    return sortedX, P

# Training
numberListLength = 5
batchSize = 32
epochs = 50
learningRate = 1e-3
trainDataset = NumberGenerator(numberListLength, 500)
trainLoader = DataLoader(trainDataset, batch_size=batchSize, shuffle=True)
network = NumberSortingNetwork(numberListLength, tau=0.5, sinkhornIterations=20)
optimizer = torch.optim.Adam(network.parameters(), lr=learningRate)

network.train()
for epoch in range(epochs):
  totalLoss = 0
  for X, sortedX, _ in trainLoader:
    sortedPrediction, _ = network(X)
    loss = F.mse_loss(sortedPrediction, sortedX)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    totalLoss += loss.item()
  print(f"Epoch {epoch+1}/{epochs}, Loss: {totalLoss/len(trainLoader):.6f}")

# Test
print("\nTesting:")
network.eval()
testDataset = NumberGenerator(numberListLength, 5, seed=420)
with torch.no_grad():
  for i in range(5):
    X, sortedX, _ = testDataset[i]
    X = X.unsqueeze(0)
    sortedPrediction, P = network(X)

    print(f"\nOriginal: {X.squeeze().tolist()}")
    print(f"Predicted sorted: {sortedPrediction.squeeze().tolist()}")
    print(f"Actual sorted: {sortedX.tolist()}")
    print(f"Permutation matrix:\n{P.squeeze().numpy().round(2)}")

Epoch 1/50, Loss: 0.090421
Epoch 2/50, Loss: 0.095273
Epoch 3/50, Loss: 0.094168
Epoch 4/50, Loss: 0.093884
Epoch 5/50, Loss: 0.095240
Epoch 6/50, Loss: 0.092008
Epoch 7/50, Loss: 0.094545
Epoch 8/50, Loss: 0.090081
Epoch 9/50, Loss: 0.093812
Epoch 10/50, Loss: 0.089087
Epoch 11/50, Loss: 0.088694
Epoch 12/50, Loss: 0.093075
Epoch 13/50, Loss: 0.091368
Epoch 14/50, Loss: 0.088067
Epoch 15/50, Loss: 0.089241
Epoch 16/50, Loss: 0.090421
Epoch 17/50, Loss: 0.088694
Epoch 18/50, Loss: 0.087907
Epoch 19/50, Loss: 0.083260
Epoch 20/50, Loss: 0.087714
Epoch 21/50, Loss: 0.080586
Epoch 22/50, Loss: 0.082756
Epoch 23/50, Loss: 0.081492
Epoch 24/50, Loss: 0.078511
Epoch 25/50, Loss: 0.080917
Epoch 26/50, Loss: 0.080953
Epoch 27/50, Loss: 0.078570
Epoch 28/50, Loss: 0.076991
Epoch 29/50, Loss: 0.078676
Epoch 30/50, Loss: 0.077668
Epoch 31/50, Loss: 0.075959
Epoch 32/50, Loss: 0.073842
Epoch 33/50, Loss: 0.079902
Epoch 34/50, Loss: 0.080849
Epoch 35/50, Loss: 0.074145
Epoch 36/50, Loss: 0.076669
E

# Bonus Content

This section is for the curious:
1. Backprop Sinkhorn iterations without autograd.
2. Converting a doubly stochastic matrix into a permutation matrix.


## Backprop Sinkhorn
For row normalization:
$$Y = X - \log(\sum_j e^{X_{ij}})$$

$$\frac{\partial Y_{ij}}{\partial X_{kl}} = \delta_{ik}\delta_{jl} - \delta_{ik} \cdot P_{il}$$

$$\nabla_X L = \nabla_Y L - P \cdot \text{rowsum}(\nabla_Y L)$$

Where $P = \text{softmax}(X) = \exp(Y)$

---

For column normalization:
$$Y = X - \log(\sum_i e^{X_{ij}})$$

$$\frac{\partial Y_{ij}}{\partial X_{kl}} = \delta_{ik}\delta_{jl} - \delta_{jl} \cdot P_{kj}$$

$$\nabla_X L = \nabla_Y L - P \cdot \text{colsum}(\nabla_Y L)$$

Where $P = \text{softmax}(X) = \exp(Y)$

---

In [6]:
import torch
torch.manual_seed(0)
#Forward
def RowNormalization(X):
  return X - torch.logsumexp(X, dim=-1, keepdim=True)
def ColNormalization(X):
  return X - torch.logsumexp(X, dim=-2, keepdim=True)
def SinkhornForward(X, iters):
  saved = []
  Z = X
  for _ in range(iters):
    Z = RowNormalization(Z)
    saved.append(("row", Z))
    Z = ColNormalization(Z)
    saved.append(("col", Z))
  return Z.exp(), saved
#Backward
def RowBackward(grad, Y):
  P = Y.exp()
  return grad - P * grad.sum(dim=-1, keepdim=True)
def Colbackward(grad, Y):
  P = Y.exp()
  return grad - P * grad.sum(dim=-2, keepdim=True)
def SinkhornBackward(grad, saved):
  # Go backwards through all saved operations
  for kind, Y in reversed(saved):
    if kind == "col":
      grad = col_backward(grad, Y)
    else:  # "row"
      grad = row_backward(grad, Y)

  return grad

# Compare with autograd
N = 8
ITERS = 20
X = torch.randn(N, N, requires_grad=True)
P, saved = SinkhornForward(X, ITERS)
loss = (P ** 2).sum()
loss.backward()
grad_autograd = X.grad.clone()

#Manual backward
#dL/dP
grad = 2 * P
#dL/dZ = dL/dP * P
grad = grad * P
#Now backprop through all the row/col normalizations
grad_manual = SinkhornBackward(grad, saved)

print("Gradient from autograd:")
print(grad_autograd)
print("\nGradient from manual:")
print(grad_manual)
print(f"\nmax error: {(grad_manual - grad_autograd).abs().max():.2e}")
print(f"mean error: {(grad_manual - grad_autograd).abs().mean():.2e}")

assert torch.allclose(grad_manual, grad_autograd, atol=1e-6)
print("\nPASS ✅")

Gradient from autograd:
tensor([[-1.3035e-02, -1.4756e-02,  6.2852e-03, -2.1250e-02,  8.1369e-02,
         -1.7832e-02, -7.0906e-03, -1.3690e-02],
        [ 3.5790e-02, -8.5389e-03,  8.3044e-03, -1.9730e-02, -1.3135e-02,
         -1.6379e-02,  4.7400e-02, -3.3712e-02],
        [-1.6980e-02, -1.0217e-02, -2.2700e-02, -4.5556e-02, -3.7852e-02,
         -5.2264e-03, -1.7684e-02,  1.5622e-01],
        [ 5.6863e-02, -1.2888e-02, -6.5493e-03, -1.7250e-02,  1.4307e-02,
         -1.8627e-02,  2.9011e-03, -1.8757e-02],
        [-7.5002e-05,  1.5610e-02,  1.2759e-03, -1.5317e-02,  1.9448e-03,
         -1.3753e-02,  3.2206e-02, -2.1891e-02],
        [-2.2516e-02,  1.0598e-02, -1.2446e-02, -2.1559e-02, -2.5946e-02,
          8.9277e-02, -5.0931e-03, -1.2316e-02],
        [-1.9403e-02,  6.3249e-02,  6.0980e-02, -2.4616e-02, -1.7323e-02,
         -1.4042e-02, -1.1054e-02, -3.7790e-02],
        [-2.0644e-02, -4.3057e-02, -3.5150e-02,  1.6528e-01, -3.3648e-03,
         -3.4178e-03, -4.1585e-02, -1.805

## Permutation Matrix From Doubly Stochastic Matrix

Conversion builds on three important ideas:


1.   Permutation matrices are orthogonal.
2.   The inverse of an orthogonal matrix is its transpose.
3. The best-corresponding permutation matrix P to a doubly-stochastic matrix X is that P whose inverse closely maps X to the identity matrix.

(Knigge, 2018) demonstrates how to approach this as a linear assignment problem using Scipy.

In [19]:
from scipy.optimize import linear_sum_assignment
from scipy.sparse import coo_matrix
import numpy as np

def matching(alpha):
  # Negate the probability matrix to serve as cost matrix. This function
  # yields two lists, the row and colum indices for all entries in the
  # permutation matrix we should set to 1.
  row, col = linear_sum_assignment(-alpha)
  # Create the permutation matrix.
  permutation_matrix = coo_matrix((np.ones_like(row), (row, col))).toarray()
  return torch.from_numpy(permutation_matrix).float()

def TraceScore(A, P):
  #same as trace(P^T A)
  return (A * P).sum()
def TestConvert():
  matrixDimension = 5
  sinkhornIterations = 200
  squareMatrix = torch.rand((matrixDimension, matrixDimension))

  doublyStochasticMatrix = SinkhornOperator(squareMatrix, sinkhornIterations)
  #Ensure result is doubly stochastic
  assert torch.allclose(doublyStochasticMatrix.sum(0), torch.ones(5), atol=1e-3)
  assert torch.allclose(doublyStochasticMatrix.sum(1), torch.ones(5), atol=1e-3)

  permutationMatrix = matching(doublyStochasticMatrix)
  bestScore = TraceScore(doublyStochasticMatrix, permutationMatrix)
  print(f"Doubly-stochastic Matrix:\n{doublyStochasticMatrix}")
  print(f"Permutation Matrix:\n{permutationMatrix}")
  for _ in range(20):
    idx = torch.randperm(matrixDimension)
    randPermutationMatrix = torch.eye(matrixDimension)[idx]
    #print(f"randPermutationMatrix : {randPermutationMatrix}")
    randScore = TraceScore(doublyStochasticMatrix, randPermutationMatrix)
    assert(bestScore > randScore)
    print(f"{bestScore}, {randScore}")



if __name__ == "__main__":
    TestConvert()

Doubly-stochastic Matrix:
tensor([[0.2233, 0.2271, 0.1743, 0.1508, 0.2245],
        [0.2339, 0.1152, 0.2471, 0.1849, 0.2189],
        [0.1772, 0.2332, 0.2279, 0.2275, 0.1343],
        [0.1576, 0.1636, 0.1837, 0.2652, 0.2299],
        [0.2081, 0.2609, 0.1670, 0.1715, 0.1924]])
Permutation Matrix:
tensor([[0., 0., 0., 0., 1.],
        [1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0.],
        [0., 0., 0., 1., 0.],
        [0., 1., 0., 0., 0.]])
1.2124379873275757, 0.9629401564598083
1.2124379873275757, 1.1265301704406738
1.2124379873275757, 1.0148423910140991
1.2124379873275757, 0.9838634729385376
1.2124379873275757, 1.06899094581604
1.2124379873275757, 0.8924182653427124
1.2124379873275757, 0.9172692894935608
1.2124379873275757, 0.986111044883728
1.2124379873275757, 0.9050251245498657
1.2124379873275757, 1.0527052879333496
1.2124379873275757, 1.1780205965042114
1.2124379873275757, 0.8967050909996033
1.2124379873275757, 0.7249221205711365
1.2124379873275757, 1.016600251197815
1.21243798